# 15 · Short-term memory and context engineering: trim, summarise, compact, cache
Level L3 · Part 3 · core · ~20 min · builds on 03, 06

## Goal
Keep a long support chat inside the window. Clear old tool output, summarise the older turns, keep the last few
turns word for word, and put the stable part of the prompt first so the model can cache it. Each step is checked
with the numbers the model returns.

## Why it matters
A model call has no memory of its own. Your code sends the whole chat again on every call. Without any management,
a fictional chat like lesson 06's resends every old order lookup on each turn, so the bill and the wait grow
with each message. A long enough chat runs into the window limit. Here is the chat with no model call. It has 11
entries. Two of them are order lookups with long scan logs. The cell counts the characters a reply reads when the
whole chat is resent.

In [1]:
import json

def scan_log(order: str) -> list[str]:
    """A fictional scan log for one parcel: 20 lines, made for this lesson."""
    return [f"day {d:02d}: scanned at sort centre {1 + (d * 3 + len(order)) % 7}" for d in range(1, 21)]

ORDER_1093 = {"status": "in transit", "days_late": 3, "scans": scan_log("1093")}
ORDER_2207 = {"status": "delivered", "days_late": 0, "scans": scan_log("2207")}
CHAT = [  # a fictional Brightside Outfitters chat, made for this lesson (not real customers)
    {"role": "user", "text": "Hi, where is my order 1093?"},
    {"role": "tool", "name": "lookup_order", "text": json.dumps(ORDER_1093)},
    {"role": "model", "text": "Order 1093 is in transit and 3 days late. Anything else I can check?"},
    {"role": "user", "text": "Can I return the jacket in that order if it is the wrong size?"},
    {"role": "model", "text": "Yes. Returns are accepted within 30 days of delivery if the item is unused, with tags on."},
    {"role": "user", "text": "And order 2207, did it arrive?"},
    {"role": "tool", "name": "lookup_order", "text": json.dumps(ORDER_2207)},
    {"role": "model", "text": "Order 2207 was delivered. Is there a problem with it?"},
    {"role": "user", "text": "Yes, it arrived in the wrong colour."},
    {"role": "model", "text": "Sorry about that. A wrong item is a return with the delivery charge refunded. Shall I start it?"},
    {"role": "user", "text": "Not yet. Remind me, what is the status of 1093 and which order had the wrong colour?"},
]

The chat is stored in SQLite (standard library), one row per turn. The window that a call reads is a query over
those rows, so trimming is one `LIMIT`. Memory here is in process; a file path would keep it between runs.

In [2]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE turns (id INTEGER PRIMARY KEY, role TEXT NOT NULL, text TEXT NOT NULL)")
db.executemany("INSERT INTO turns (role, text) VALUES (?, ?)", [(e["role"], e["text"]) for e in CHAT])


def window_from_db(n: int) -> list[dict]:
    """The last n turns, oldest first. The window is a query, not a copy of the whole chat."""
    rows = db.execute("SELECT role, text FROM turns ORDER BY id DESC LIMIT ?", (n,)).fetchall()
    return [{"role": r, "text": t} for r, t in reversed(rows)]


print("turns stored:", db.execute("SELECT COUNT(*) FROM turns").fetchone()[0], "· last 2 roles:",
      [e["role"] for e in window_from_db(2)])

turns stored: 11 · last 2 roles: ['model', 'user']


Now count what each reply reads when the whole chat is resent.

In [3]:
def chars_sent(chat: list[dict]) -> int:
    """Characters a reply reads when the whole chat is resent. Characters are a rough stand-in for tokens."""
    return sum(len(e["text"]) for e in chat)


print(f"{'turn':>4} {'characters sent':>16}")
for n in range(1, len(CHAT) + 1):
    print(f"{n:>4} {chars_sent(CHAT[:n]):>16,}")
print(f"total read over the chat: {sum(chars_sent(CHAT[:n]) for n in range(1, len(CHAT) + 1)):,} characters")

turn  characters sent
   1               27
   2              798
   3              866
   4              928
   5            1,017
   6            1,047
   7            1,817
   8            1,870
   9            1,906
  10            2,001
  11            2,085
total read over the chat: 14,362 characters


**What the table shows:** each turn reads everything before it. The two tool results are most of the characters, and
they are needed only for the turns that talk about those orders. The total line counts every turn rereading the past,
so it is far bigger than the chat itself (2,085 characters). **What it makes easy:** one place to decide what each
call reads. **When you don't need it:** a chat of two or three turns fits in the window, and nothing here is needed.

## The idea
In this lesson, **short-term memory** is the list of messages you resend on each call. **Context engineering** is the
choice of what goes in that list. Four levers do most of the work. **Trimming** drops the oldest entries. **Clearing**
replaces old tool results with a short stub. **Summarising** (also called **compaction**) turns older entries into one
short note written by the model. **Caching** makes a stable beginning of the prompt cheaper to send again. You check
each lever with the usage numbers from the response, not with a guess. **Just-in-time retrieval** means keeping an id
in the chat and looking the data up when it is needed (the stub in step 1 does this). Lesson 17 shows retrieval.

Analogy: a support agent clears finished paperwork off the desk, writes a one-page handover note, keeps the last few
letters open, and leaves the policy binder open at the same page.

## Picture
```mermaid
flowchart LR
  H[Full chat: every entry] --> C[Clear old tool results: no call]
  C --> S[Summarise the older entries: one model call]
  S --> W[Window: the summary plus the last 3 entries word for word]
  W --> M[Model call]
  P[Cache: the rule text and the instruction, first] --> M
```

## Step by step
**Step 1 · clear the old tool results (no call).** A tool result can be long, and the answer usually needs only its
status. The function below keeps the last few entries whole and replaces each older tool result with a stub. The stub
says what was removed, so the model knows to ask again if it needs the detail. Clearing and summarising change only
the copy you send; the DB rows stay whole.

In [4]:
STUB = "[cleared: an old order lookup. Look it up again if the detail is needed.]"
KEEP = 3  # the last entries that stay word for word


def clear_tool_results(chat: list[dict], keep_last: int) -> list[dict]:
    """Replace each tool result older than the last keep_last entries with a short stub."""
    cut = len(chat) - keep_last
    return [dict(e, text=STUB) if e["role"] == "tool" and i < cut else e for i, e in enumerate(chat)]


cleared = clear_tool_results(CHAT, KEEP)
print("characters before:", chars_sent(CHAT), "· after clearing:", chars_sent(cleared))

characters before: 2085 · after clearing: 690


**Step 2 · the client (setup).** The key is read from `.env` and never typed here. While a recording replays, a
placeholder is used and no key is sent. `llm_replay` is a course helper, not part of the SDK: it saves each model call
once, so later runs replay the saved answers. The model is the same as lessons 02 to 06.

In [5]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("15_short-term-memory-context")

from google import genai
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model, the same as lessons 02 to 06
client = genai.Client(  # in your own project: put your key in .env as GOOGLE_CLOUD_API_KEY and keep vertexai=True
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx
)

LLM mode: replay (15_short-term-memory-context.yaml)


**Step 3 · summarise the older entries (one model call).** Everything before the last three entries goes to the
model with one instruction: keep the order numbers, statuses, amounts and open requests. The model's answer is the
summary. Summarising is lossy, so the check below asks whether the facts that matter survived. Clearing ran first, so
the summariser never sees the old scan logs. That is the trade-off: fewer characters, and no detail that only a tool
result held.

In [6]:
RULES = ("Summarise this support chat so the agent can carry on. Keep order numbers, statuses, dates, amounts "
         "and open requests. Leave out pleasantries. At most 60 words.")
transcript = "\n".join(f"{e['role']}: {e['text']}" for e in cleared[:-KEEP])
resp = client.models.generate_content(
    model=MODEL, contents=transcript,
    config=types.GenerateContentConfig(system_instruction=RULES, temperature=0, max_output_tokens=1024))
summary = resp.text
print(summary)
print("input tokens for the summary call:", f"{resp.usage_metadata.prompt_token_count:,}")
print({k: (k in summary) for k in ("1093", "in transit", "2207")})

Order 1093 is in transit (3 days late). Customer inquired about return policy for a jacket in this order; confirmed returns accepted within 30 days of delivery if unused with tags. Order 2207 is confirmed delivered. Agent needs to address if there are any issues with order 2207.
input tokens for the summary call: 196
{'1093': True, 'in transit': True, '2207': True}


**Step 4 · answer from the compacted chat (one model call).** The window is now the summary, which goes in the system
instruction, plus the last three entries word for word. The question at the end needs both: the status of 1093 is in
the summary, and the wrong colour is in the recent turns. The window is smaller than the whole chat. The usage
record shows the input tokens the call really read. Clearing and summarising change only the copy you send; the DB rows
stay whole.

In [7]:
summary = globals().get("summary", "(summary skipped: run the step above first)")  # local fallback
ROLE = {"user": "user", "model": "model", "tool": "user"}  # the SDK calls the assistant role "model"


def to_contents(entries: list[dict]) -> list[types.Content]:
    return [types.Content(role=ROLE[e["role"]], parts=[types.Part(text=e["text"])]) for e in entries]


window = window_from_db(KEEP)  # the same last entries as cleared[-KEEP:], read from SQLite
SYSTEM = f"You are the Brightside Outfitters support desk. Summary of the earlier chat: {summary}"
print(f"characters in the window: {chars_sent(window) + len(summary):,} (the whole chat: {chars_sent(CHAT):,})")
resp = client.models.generate_content(
    model=MODEL, contents=to_contents(window),
    config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0, max_output_tokens=1024))
print(resp.text)
print("input tokens for the answer call:", f"{resp.usage_metadata.prompt_token_count:,}")

characters in the window: 494 (the whole chat: 2,085)
I apologize for the confusion. Let me clarify the status of both orders for you:

*   **Order 1093:** This order is currently **in transit** and is unfortunately running 3 days behind schedule. As we discussed, once this arrives, you have 30 days to return the jacket if it is unused and the tags are still attached.
*   **Order 2207:** This is the order that **arrived in the wrong color**. 

I am very sorry that you received the incorrect item for order 2207. Since you aren't ready to start the return process just yet, please let me know whenever you are ready, and I will be happy to assist you with the return label and ensure your shipping costs are fully refunded. 

Is there anything else I can help you with regarding these orders today?
input tokens for the answer call: 138


**What to look for:** the answer should say that 1093 is in transit and that 2207 arrived in the wrong colour. Both
facts are there: 1093 came from the summary and the colour from the last turns. The extra wording ("I apologize",
the questions at the end) is normal model chatter, not a bug.

**Step 5 · cache the stable prefix (two model calls).** Caching works on the start of the prompt. When a new call
begins with the same long start as an earlier one, the service can reuse the work it did for that start, and the
usage record reports the reused tokens as `cached_content_token_count`. On Gemini 2.5 and newer models this is
**implicit**: no setting to change, on by default, and best effort, so the service may or may not hit. Explicit
caching (you create a cache object, with a TTL, and pass its name) also exists, but it is not available with this
key, so this lesson uses implicit caching only.

The long, unchanging start is the FTC's 30-day shipping rule. It is about 5,000 words, and it goes first.

In [8]:
FTC_RULE = open("../data/ftc_30_day_rule.txt", encoding="utf-8").read()
print(FTC_RULE[:160].replace("\n", " "), "...")
print(f"{len(FTC_RULE.split()):,} words · {len(FTC_RULE):,} characters")

How to Comply With the Rule  The following information will help you comply with the Rule.  What You Should Know Before You Make a Shipment Representation  When ...
5,029 words · 30,156 characters


Cached input tokens are billed at a discount. The Gemini price list gives 90% for Gemini models when this lesson was
written (check https://ai.google.dev/gemini-api/docs/pricing the day you read this). Each model has a minimum prompt
size for caching, listed on https://ai.google.dev/gemini-api/docs/caching. This model's minimum was not confirmed
there, so read the cached-token count instead of assuming. Caching works in blocks, so a few tokens at the end are
not cached. This prompt is about 6,200 tokens. Next, the questions. Both send the same start, with the question last.

In [9]:
RULE_RULES = "Answer from the rule text only, in at most two sentences."


def ask_rule(question: str):
    """The same long text first, the question last, and the same settings on every call."""
    return client.models.generate_content(
        model=MODEL,
        contents=[types.Content(role="user", parts=[types.Part(text=FTC_RULE), types.Part(text=question)])],
        config=types.GenerateContentConfig(system_instruction=RULE_RULES, temperature=0, max_output_tokens=1024))


def report(name: str, resp) -> None:
    u = resp.usage_metadata
    print(f"{name}: input {u.prompt_token_count:,} tokens · from cache {u.cached_content_token_count or 0:,}")
    print("  ", resp.text.strip())


report("question 1", ask_rule("If the seller cannot ship in time, what must it do once the promised time has passed?"))
report("question 2", ask_rule("What is the longest a delay notice can push the shipping date when no time was promised?"))
llm_replay.stop()  # the last model call of this notebook

question 1: input 6,237 tokens · from cache 6,108
   If you cannot ship the order or provide the required delay notification within the originally promised time (or 30 days if no time was promised), you must cancel the order and make a prompt refund. You are not permitted to notify the customer after this time has already passed.
question 2: input 6,236 tokens · from cache 6,108
   If no shipment time was promised, the Rule requires you to ship or provide a delay notice within 30 days. Consequently, no notification to the customer can take longer than that 30-day period.


**Read the result:** both questions read the same start. The cached count shows how much of the input was reused:
about 6,100 of about 6,200 tokens, so only the question was new. In this recording question 1 also hit, because the
same start had been sent in earlier calls. On your own run question 1 may show 0 and question 2 may hit. Compare the
two counts. A hit is never promised: the count is the evidence, and zero means the start was not reused. The answers
should say that the seller must cancel the order and make a prompt refund, and that, when no time was promised, the
seller must ship or send the delay notice within 30 days. The second answer is about when the notice is due, not how
far it may push the date.

## At work
- Long support chats are a top cost. Teams clear old tool output and summarise older turns, and vendors now offer
  server-side compaction too (check each vendor's docs for the current name). Lost facts are a known failure in long
  runs → lesson 25 (loops, lost context and guards).
- Cache hits appear as their own count in the usage record. Trace tools that add them to the input count make cache
  savings look like full price → lesson 29 (traces) and lesson 33 (cost caps).
- LangChain/LangGraph ship the same ideas for their own message types (trim_messages, summarisation middleware) → lesson 19.
- Long-term memory (facts kept across sessions) is a separate store, not a longer window → lesson 16.

## Common mistakes
1. **A window that starts with a tool result.** Slicing the chat by count can cut a request away from its result. A
   model cannot match a result to the call that asked for it. The check below raises before any call is made.

In [10]:
def check_window(entries: list[dict]) -> list[dict]:
    """A window must not start with a tool result: the request it answers was cut away."""
    if entries and entries[0]["role"] == "tool":
        raise ValueError("the window starts with a tool result, with no request before it")
    return entries


check_window(CHAT[6:])  # entry 6 is the 2207 lookup result; its request (entry 5) was cut away

ValueError: the window starts with a tool result, with no request before it

2. **Question first, document second.** Caching matches from the start of the prompt. Compare the two orders with no
   call: the first character that differs ends the shared start.

In [11]:
import os  # the standard library's common-prefix helper works on any two strings

question_first = "Question: what is the delay rule? " + FTC_RULE
rule_first = FTC_RULE + " Question: what is the delay rule?"
other_question = "Question: what is the return window? " + FTC_RULE
shared = lambda a, b: len(os.path.commonprefix([a, b]))  # characters the two prompts share from the start
print("question first, two questions share:", shared(question_first, other_question), "characters")
print("rule first, two questions share:    ", shared(rule_first, FTC_RULE + " Question: what is the return window?"), "characters")

question first, two questions share: 22 characters
rule first, two questions share:     30179 characters


Read the output: the rule-first prefix is longer than the rule itself (30,156 characters). That is because the
question's first words, "Question: what is the", are shared by both prompts too. Only the part after them differs.

3. **Summary and old turns both sent.** After a summary, keep the summarised entries out of the window. Otherwise the
   cost goes down on paper and up in the bill. The cell below counts both windows with no call. The summary is the one
   from step 3.

In [12]:
summary_text = globals().get("summary", "")  # from step 3 (empty if that step was skipped)
print(f"mistake, summary + whole chat: {len(summary_text) + chars_sent(CHAT):,} characters")
print(f"fix, summary + last {KEEP} turns: {len(summary_text) + chars_sent(window_from_db(KEEP)):,} characters")

mistake, summary + whole chat: 2,364 characters
fix, summary + last 3 turns: 494 characters


## Try it
A window that starts on a tool result is a bad window, so write `safe_window(chat, n)`. It returns the last `n` entries.
If the first of them is a tool result, it moves forward one entry at a time until the window starts with a
non-tool entry. Check it with `n=5` (which starts on the 2207 result) and with `n=3`. No model call is needed.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
def safe_window(chat: list[dict], n: int) -> list[dict]:
    start = len(chat) - n
    while chat[start]["role"] == "tool":
        start += 1
    return chat[start:]


print("n=5 starts with:", safe_window(CHAT, 5)[0]["role"], "· entries:", len(safe_window(CHAT, 5)))
print("n=3 starts with:", safe_window(CHAT, 3)[0]["role"], "· entries:", len(safe_window(CHAT, 3)))

n=5 starts with: model · entries: 4
n=3 starts with: user · entries: 3


## Recap
- Each call reads everything you send. The chat's size, not its last message, sets the cost of every turn.
- Clear old tool results first, then summarise the older entries, and keep the last few turns word for word.
- Check a summary for the facts that matter, and check the window for a tool result with no request before it.
- Put the large, stable text first and the question last. Read the cached-token count to see whether it worked.

**Interview:** "A support chat runs to 300 turns and the bill keeps climbing. What do you do?" Strong answer: "I clear
old tool output, summarise the older turns into a note that keeps the ids and open requests, and keep the last few
turns whole. I put the stable policy text first so it caches, and I read the cached-token count to check it."